# 4 · ReAct — Look It Up, Don't Guess

**The problem:** the model does not know your company's data.
Ask it about a specific customer order and it will **invent** the details —
dates, amounts, everything — and sound completely certain while doing it.

**The fix:** ReAct (short for **Reason + Act**). Give the model a small set of
lookup tools and run a loop: *think about what you need → look it up →
read what actually came back → think again → answer.*

This is exactly how the chatbot in the `simple_agent_chatbot` project works —
same LangChain function, same pattern, just smaller.

**The scenario:**

> You run a customer service desk. A customer asks:
> *"Can I get a refund on my mixer grinder, order A-4471?"*
>
> The truth (in our small database below): it was delivered on **2 June**,
> the refund window for kitchen appliances is **14 days**, and today is **22 August**.
> So the honest answer is **no — the window closed months ago**.

In [ ]:
# ---- Step 0: check the kernel, then install what is missing ----
# Run this first. It works on Colab, on a fresh laptop,
# and it tells you plainly if the notebook is running the wrong Python.

import importlib.util, subprocess, sys

if sys.version_info < (3, 10):
    print("STOP - this notebook needs Python 3.10 or newer.")
    print("This kernel is Python", sys.version.split()[0], "at", sys.executable)
    print()
    print("Fix it like this:")
    print("  In Jupyter / VS Code : Kernel > Change Kernel, and pick the one from")
    print("                         structured_prompting/.venv")
    print("  On Colab             : Runtime > Restart session, then run this cell again")
    raise SystemExit("Wrong Python version - see the message above.")

REQUIRED = [
    ("langchain", "langchain==1.3.0"),
    ("langchain_openai", "langchain-openai==1.2.1"),
    ("dotenv", "python-dotenv==1.2.2"),
]

missing = [pkg for mod, pkg in REQUIRED if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    print("(a minute the first time, nothing the next time)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done.")
else:
    print("All libraries already here.")

print("Python", sys.version.split()[0], "at", sys.executable)


In [ ]:
# ---- Step 1: imports and your OpenAI key ----
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()  # reads the .env file next to this notebook, if there is one

if not os.getenv("OPENAI_API_KEY"):
    try:
        # Google Colab: add OPENAI_API_KEY in the Secrets panel (the key icon, left)
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        # Last resort: type it here. It is not saved anywhere.
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI key: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
model = ChatOpenAI(model=MODEL_NAME, temperature=0)

print("Setup done. Using model:", MODEL_NAME)


In [ ]:
# ---- Our tiny "company database" ----
# In real life these would be real systems. Here, two small dictionaries.

ORDERS = {
    "A-4471": {
        "product": "mixer grinder",
        "category": "kitchen appliance",
        "delivered_on": "2026-06-02",
        "status": "delivered",
    },
    "A-5810": {
        "product": "office chair",
        "category": "furniture",
        "delivered_on": "2026-08-15",
        "status": "delivered",
    },
}

REFUND_POLICY = {
    "kitchen appliance": "Refund within 14 days of delivery, product unused.",
    "furniture": "Refund within 30 days of delivery.",
}

TODAY = "2026-08-22"

---
## Part 1 · The wrong way — no tools, just "think step by step"

The model has never seen order A-4471. Watch what it does anyway.

In [ ]:
wrong_prompt = """A customer asks: "Can I get a refund on my mixer grinder,
order A-4471?"

Think step by step, then give the customer an answer."""

print(model.invoke(wrong_prompt).content)

Read the reply. It will produce neat, numbered steps — and somewhere in them
it **invents** a purchase date, or a policy, or both. Every step looks
professional. Every fact is made up.

This is the most dangerous kind of wrong answer: it *looks checked*.
A customer service agent who sends this reply has just promised (or refused)
a refund based on fiction.

> **Lesson:** step-by-step thinking cannot create a fact the model never had.
> For facts, the model has to go and look.

---
## Part 2 · The right way — give it tools and rules

Two tools. Each one is a plain Python function with a `@tool` sticker on top.
The text under each function (the docstring) is how the model decides
**when** to use it — so we write it carefully:
what it is for, and what an empty result means.

In [ ]:
@tool
def get_order(order_id: str) -> str:
    """Look up one customer order by its order id (format: A-followed by digits).
    Returns the product, category, delivery date and status.
    If the reply is NOT FOUND, the order does not exist - that is a real answer,
    do not try again with the same id."""
    order = ORDERS.get(order_id)
    if order is None:
        return f"NOT FOUND: there is no order {order_id}."
    return (f"Order {order_id}: {order['product']} ({order['category']}), "
            f"delivered on {order['delivered_on']}, status: {order['status']}.")

@tool
def get_refund_policy(category: str) -> str:
    """Look up the refund policy for a product category,
    for example 'kitchen appliance' or 'furniture'."""
    policy = REFUND_POLICY.get(category)
    if policy is None:
        return f"NOT FOUND: no policy for category '{category}'."
    return f"Policy for {category}: {policy}"

In [ ]:
# The rules. Notice they are written as bans, not suggestions -
# "never state a fact you did not look up" works, "use tools when helpful" does not.

RULES = f"""You are a customer service assistant. Today's date is {TODAY}.

Rules you must follow:
- NEVER state an order detail, a date or a policy that you did not
  get from a tool in this conversation.
- If a tool cannot give you a fact, say "I could not verify this" -
  do not guess and do not estimate.
- Answer the customer politely, with the real dates in your answer."""

agent = create_agent(model, [get_order, get_refund_policy], system_prompt=RULES)

question = 'Can I get a refund on my mixer grinder, order A-4471?'
result = agent.invoke({"messages": [HumanMessage(content=question)]})

### Watch the loop — think, look up, read, answer

The result contains every message the agent produced.
Let's print them as a simple trace.

In [ ]:
def print_trace(result):
    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "HumanMessage":
            print("QUESTION :", message.content)
        elif kind == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("LOOK UP  :", call["name"], call["args"])
        elif kind == "ToolMessage":
            print("RESULT   :", message.content)
        elif kind == "AIMessage":
            print("ANSWER   :", message.content)
        print()

print_trace(result)

You should see the full loop:

1. **LOOK UP** `get_order` with A-4471 → **RESULT**: delivered 2 June
2. **LOOK UP** `get_refund_policy` for kitchen appliances → **RESULT**: 14 days
3. **ANSWER**: no — delivered 2 June, the 14-day window closed in mid-June

Compare this with Part 1. The answer is shorter and less clever —
and it is the only version a company could actually send to a customer,
because **every fact in it came from a lookup you can see in the trace**.

### What about an order that does not exist?

The professional answer to "I couldn't find it" is to *say so* — not to guess.

In [ ]:
result2 = agent.invoke({"messages": [HumanMessage(
    content="Can I return my laptop? Order Z-9999.")]})

print_trace(result2)

The lookup comes back NOT FOUND, and the agent says it cannot find that order
and asks the customer to check the number. No invented laptop, no invented policy.

> **"I could not verify this" is a professional answer. A confident guess is not.**

---
## Summary

1. **What it is:** a loop — think, use one tool, read the real result, repeat, answer.
2. **The tools' descriptions matter:** they are how the model chooses. Say what the
   tool is for and what an empty result means.
3. **Write the rules as bans:** "never state a fact you did not look up."
4. **Read the trace, not just the answer.** A wrong answer and a right answer can
   look identical — the trace is where you see the difference.

**Try it yourself:** ask about order A-5810 (the office chair, delivered 15 August).
Furniture has a 30-day window and today is 22 August — the agent should say **yes**.